### Script for Forehand Depth JSON

In [4]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')

# TODO: Update these variables when running notebook
default_match_file = "Shot_Visuals_CassiusChinlund_GregGamal.csv"
if os.path.exists(os.path.join("..", "match-csvs", default_match_file)):
    path = os.path.join("..", "match-csvs", default_match_file)
else:
    path = os.path.join("data", "match-csvs", default_match_file)

# remember to change the player name to the current matches' UCLA player name
ucla_player = "Cassius Chinlund"
output_filename = "avg_forehand_depth.json"

# You can pass one match path or multiple paths.
# Example for multiple:
# paths = [
#     "../match-csvs/Shot_Visuals_RudyQuan_IiroVasa.csv",
#     "../match-csvs/Shot_Visuals_RudyQuan_AristotelisThanos.csv"
# ]
paths = [path]

# Adjusted output path for JSONs (works whether cwd is repo root or data/notebooks)
if os.path.isdir("data/json"):
    output_dir = os.path.join(os.getcwd(), "data/json")
elif os.path.isdir("../json"):
    output_dir = os.path.join(os.getcwd(), "../json")
else:
    output_dir = os.path.join(os.getcwd(), "data", "json")
os.makedirs(output_dir, exist_ok=True)


def _clean_value(value, fallback="Unknown"):
    if pd.isna(value):
        return fallback
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return fallback
    return text


def _resolve_match_path(match_path):
    if os.path.exists(match_path):
        return match_path

    file_name = os.path.basename(match_path)
    candidates = [
        os.path.join("data", "match-csvs", file_name),
        os.path.join("..", "match-csvs", file_name)
    ]

    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate

    raise FileNotFoundError(f"Could not find match CSV: {match_path}")


def _build_match_record(events, player, source_path):
    required_cols = {'shotHitBy', 'shotFhBh', 'shotLocationY'}
    missing = [c for c in required_cols if c not in events.columns]
    if missing:
        raise ValueError(f"Missing required columns in {source_path}: {missing}")

    forehands = events[
        (events['shotHitBy'] == player) &
        (events['shotFhBh'].astype(str).str.lower() == 'forehand')
    ].copy()

    forehands['depth'] = pd.to_numeric(forehands['shotLocationY'], errors='coerce').abs()
    forehands = forehands[forehands['depth'].notna()]

    if forehands.empty:
        return None

    first_row = events.iloc[0]

    player1 = _clean_value(first_row.get('player1Name'), fallback='')
    player2 = _clean_value(first_row.get('player2Name'), fallback='Unknown Opponent')
    if player1 == player and player2 != 'Unknown Opponent':
        opponent = player2
    elif player2 == player and player1 != '':
        opponent = player1
    else:
        opponent = player2 if player2 != '' else _clean_value(first_row.get('opponentTeam'), 'Unknown Opponent')

    depth_series = forehands['depth']

    return {
        'matchId': f"vs {opponent}",
        'sourceFile': os.path.basename(source_path),
        'opponent': opponent,
        'date': _clean_value(first_row.get('Date')),
        'round': _clean_value(first_row.get('Round')),
        'surface': _clean_value(first_row.get('Surface')),
        'forehandCount': int(len(depth_series)),
        'averageDepth': round(float(depth_series.mean()), 1),
        'medianDepth': round(float(depth_series.median()), 1),
        'minDepth': round(float(depth_series.min()), 1),
        'maxDepth': round(float(depth_series.max()), 1),
        'depthValues': [round(float(x), 1) for x in depth_series.tolist()]
    }


def average_forehand_depth(player, match_paths, output_name="avg_forehand_depth.json"):
    records = []

    for match_path in match_paths:
        resolved_path = _resolve_match_path(match_path)
        events = pd.read_csv(resolved_path)
        record = _build_match_record(events, player, resolved_path)
        if record is not None:
            records.append(record)

    if not records:
        raise ValueError("No valid forehand depth data found for the selected player/match paths.")

    all_depths = [value for record in records for value in record['depthValues']]
    all_depths_series = pd.Series(all_depths, dtype='float64')

    payload = {
        'player': player,
        'metricDefinition': 'Depth is calculated as abs(shotLocationY), where higher values indicate deeper forehands.',
        'matches': records,
        'overall': {
            'matchCount': int(len(records)),
            'totalForehands': int(len(all_depths)),
            'averageDepth': round(float(all_depths_series.mean()), 1),
            'medianDepth': round(float(all_depths_series.median()), 1),
            'q1': round(float(all_depths_series.quantile(0.25)), 1),
            'q3': round(float(all_depths_series.quantile(0.75)), 1),
            'iqr': round(float(all_depths_series.quantile(0.75) - all_depths_series.quantile(0.25)), 1)
        }
    }

    out_path = os.path.join(output_dir, output_name)
    with open(out_path, 'w') as f:
        json.dump(payload, f, separators=(',', ':'))

    print(f"Wrote {out_path}")
    print(f"Player: {player}")
    print(f"Forehands: {len(all_depths)}")


# run the pipeline
average_forehand_depth(ucla_player, paths, output_filename)


Wrote /Users/pr3s10/Desktop/tc_analytics/data/notebooks/../json/avg_forehand_depth.json
Player: Cassius Chinlund
Forehands: 205


### Script for Forehand-Only JSON

In [6]:
def forehands_only(player, match_path, output_dir, output_name="forehands_only.json", output_dist_name="forehands_only_dist.json"):
    events = pd.read_csv(match_path)
    events["pointWonBy"] = events.groupby("pointNumber")["pointWonBy"].bfill()
    events["isError"] = (
        (events["isErrorWideR"] == 1)
        | (events["isErrorWideL"] == 1)
        | (events["isErrorNet"] == 1)
        | (events["isErrorLong"] == 1)
    )

    shot_side = events["shotFhBh"].astype(str).str.strip().str.lower()
    is_slice = pd.to_numeric(events["isSlice"], errors="coerce").fillna(0).eq(1)

    forehands = events[
        (events["shotHitBy"] == player)
        & (events["shotInRally"] != 1)
        & (shot_side == "forehand")
        & (~is_slice)
    ][
        [
            "pointStartTime", "shotHitBy", "shotContactX", "shotContactY",
            "shotLocationX", "shotLocationY", "pointWonBy", "isWinner",
            "shotFhBh", "isError", "isErrorNet", "side", "isSlice"
        ]
    ].dropna(subset=["pointWonBy"]).copy()

    mask_bottom_half = (forehands["shotLocationY"] < 0) & (forehands["shotContactY"] > 0)
    mask_near_net = (
        (forehands["shotLocationY"] <= 50)
        & (forehands["shotContactY"] > 0)
        & (forehands["isErrorNet"] == 1)
    )

    forehands.loc[mask_bottom_half, "shotContactX"] *= -1
    forehands.loc[mask_bottom_half, "shotLocationX"] *= -1
    forehands.loc[mask_bottom_half & (forehands["shotContactY"] > 0), "shotContactY"] *= -1
    forehands.loc[mask_bottom_half, "shotLocationY"] = forehands.loc[mask_bottom_half, "shotLocationY"].abs()

    forehands.loc[mask_near_net & ~mask_bottom_half, "shotContactX"] *= -1
    forehands.loc[mask_near_net & ~mask_bottom_half, "shotLocationX"] *= -1
    forehands.loc[mask_near_net & ~mask_bottom_half, "shotContactY"] *= -1

    mask = (forehands["shotLocationY"] != 0) & (forehands["isErrorNet"] == 1)
    adjust_up = mask & (forehands["shotLocationX"] <= forehands["shotContactX"])
    adjust_down = mask & (forehands["shotLocationX"] > forehands["shotContactX"])

    forehands.loc[adjust_up, "shotLocationX"] += forehands.loc[adjust_up, "shotLocationY"]
    forehands.loc[adjust_down, "shotLocationX"] -= forehands.loc[adjust_down, "shotLocationY"]
    forehands.loc[adjust_up, "shotContactX"] += forehands.loc[adjust_up, "shotLocationY"]
    forehands.loc[adjust_down, "shotContactX"] -= forehands.loc[adjust_down, "shotLocationY"]
    forehands.loc[adjust_up, "shotLocationY"] = 0
    forehands.loc[adjust_down, "shotLocationY"] = 0

    forehands = forehands[(forehands["shotLocationX"] >= -300) & (forehands["shotLocationX"] <= 300)]
    forehands["shotFhBh"] = "Forehand"
    forehands["isSlice"] = 0
    forehands["fhBhFiltered"] = True
    forehands["shotType"] = "Forehand (Non-Slice)"
    forehands["width"] = forehands["shotLocationX"].apply(
        lambda x: "left" if x <= -52.5 else "mid" if -52.5 < x < 52.5 else "right"
    )

    distribution = forehands.groupby("width").apply(
        lambda df: pd.Series({
            "freq": len(df),
            "win_percentage": int((df["pointWonBy"] == df["shotHitBy"]).mean() * 100)
        })
    ).reset_index()

    if not distribution.empty:
        max_win_percentage = distribution["win_percentage"].max()
        min_win_percentage = distribution["win_percentage"].min()
        distribution["maxMin"] = distribution["win_percentage"].apply(
            lambda x: "max" if x == max_win_percentage else "min" if x == min_win_percentage else "no"
        )
        distribution["win_percentage"] = distribution["win_percentage"].astype(str) + "%"

    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, output_name), "w") as f:
        json.dump(json.loads(forehands.to_json(orient="records")), f, separators=(",", ":"))

    with open(os.path.join(output_dir, output_dist_name), "w") as f:
        json.dump(json.loads(distribution.to_json(orient="records")), f, separators=(",", ":"))

    print(f"Player: {player}")
    print(f"Wrote {os.path.join(output_dir, output_name)}")
    print(f"Wrote {os.path.join(output_dir, output_dist_name)}")
    print(f"Forehands (non-slice): {len(forehands)}")


forehands_only(ucla_player, path, output_dir)


Player: Cassius Chinlund
Wrote /Users/pr3s10/Desktop/tc_analytics/data/notebooks/../json/forehands_only.json
Wrote /Users/pr3s10/Desktop/tc_analytics/data/notebooks/../json/forehands_only_dist.json
Forehands (non-slice): 197
